1. recovery_rate → 실패 복구 능력
2. calls_to_recovery → 복구 속도
3. success_rate_by_state_action → policy 타당성
4. success_per_call → 비용 효율성
5. stagnation vs plan 전환 효과

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# ── 한국어 폰트 설정 ──
nanum = [f.fname for f in fm.fontManager.ttflist if 'NanumGothic' in f.name]
noto  = [f.fname for f in fm.fontManager.ttflist if 'NotoSansCJK' in f.name or 'Noto Sans CJK' in f.name]

if nanum:
    plt.rcParams['font.family'] = 'NanumGothic'
elif noto:
    plt.rcParams['font.family'] = 'Noto Sans CJK KR'
else:
    print('⚠️ 한국어 폰트를 찾을 수 없습니다. 기본 폰트를 사용합니다.')

plt.rcParams['axes.unicode_minus'] = False

# ── 경로 설정 ──
NOTEBOOK_DIR  = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))
RESULTS_ROOT  = os.path.join(PROJECT_ROOT, 'results', 'phase2_qwen')

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'RESULTS_ROOT : {RESULTS_ROOT}')

# ── 스타일 설정 ──
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── 상수 ──
DATASETS = ['humaneval']
# METHODS  = ['single', 'repair','code_then_plan','code_then_replan','code_then_plan_repair','rpe_policy']
METHODS  = ['rpe_policy']

METHOD_LABELS = {
    'single'   : 'Single Shot',
    'repair'   : 'Repair Loop',
    'planner_coder' : 'Planner-Coder',
    'code_then_plan' : 'Code-Then-Plan',
    'code_then_replan' : 'Code-Then-Replan',
    'code_then_plan_repair' : 'Code-Then-Plan+Repair',
    'rpe_policy' : 'Policy Loop'
}

# ── 색상 팔레트 ──
METHOD_COLORS = {
    'single'        : '#4C72B0',
    'repair'       : '#55A868',
    'planner_coder'        : '#C44E52',
    'code_then_plan' : '#8172B2',
    'code_then_replan' : '#9EDDFF',
    'code_then_plan_repair' : '#DD8452',
    'rpe_policy' : '#FFA500'
}

PROJECT_ROOT : /home/dibaeck/workspace/project_IR_focus_sLM_orchestration
RESULTS_ROOT : /home/dibaeck/workspace/project_IR_focus_sLM_orchestration/results/phase2_qwen


---
## 1. 데이터 로드

In [2]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# ── 결과 로드 ──
summaries          = {}   # (dataset, method) -> summary dict
analyses           = {}   # (dataset, method) -> analysis dict
step_logs_all      = {}   # (dataset, method) -> list of step records
traj_logs_all      = {}   # (dataset, method) -> list of trajectory records
failure_examples_all = {} # (dataset, method) -> dict of failure examples

found = []
missing = []

for dataset in DATASETS:
    for method in METHODS:
        base = os.path.join(RESULTS_ROOT, dataset, method)
        summary_path = os.path.join(base, 'summary.json')
        if os.path.exists(summary_path):
            key = (dataset, method)
            summaries[key]     = load_json(summary_path)
            step_logs_all[key] = load_jsonl(os.path.join(base, 'step_logs.jsonl'))
            traj_logs_all[key] = load_jsonl(os.path.join(base, 'trajectory_logs.jsonl'))
            
            fe_path = os.path.join(base, 'failure_examples.json')
            if os.path.exists(fe_path):
                failure_examples_all[key] = load_json(fe_path)
            
            found.append(f'{dataset}/{method}')
        else:
            missing.append(f'{dataset}/{method}')

print(f'✅ 로드 성공 ({len(found)}개):', ', '.join(found))
if missing:
    print(f'⚠️  결과 없음 ({len(missing)}개):', ', '.join(missing))

✅ 로드 성공 (1개): humaneval/rpe_policy


In [3]:
def get_dfs(dataset='humaneval', method='rpe_policy'):
    key = (dataset, method)
    step_df = pd.DataFrame(step_logs_all[key])
    traj_df = pd.DataFrame(traj_logs_all[key])
    return step_df, traj_df

step_df, traj_df = get_dfs('humaneval', 'rpe_policy')

print('step_df:', step_df.shape)
print('traj_df:', traj_df.shape)

step_df: (562, 34)
traj_df: (164, 32)


In [4]:
initial_failed = traj_df[traj_df['initial_failed'] == True].copy()

n_total = len(traj_df)
n_initial_failed = len(initial_failed)
n_recovered = (initial_failed['final_status'] == 'PASS').sum()
n_unresolved = (initial_failed['final_status'] != 'PASS').sum()

recovery_rate = n_recovered / n_initial_failed if n_initial_failed else 0
unresolved_rate = n_unresolved / n_initial_failed if n_initial_failed else 0

pd.DataFrame([{
    'total_problems': n_total,
    'initial_failed': n_initial_failed,
    'recovered': n_recovered,
    'unresolved': n_unresolved,
    'recovery_rate': recovery_rate,
    'unresolved_rate': unresolved_rate,
}])

,total_problems,initial_failed,recovered,unresolved,recovery_rate,unresolved_rate
0,164,55,42,13,0.763636,0.236364


In [5]:
recovered = initial_failed[initial_failed['final_status'] == 'PASS'].copy()

pd.DataFrame([{
    'avg_calls_to_recovery': recovered['calls_to_recovery'].mean(),
    'median_calls_to_recovery': recovered['calls_to_recovery'].median(),
    'min_calls_to_recovery': recovered['calls_to_recovery'].min(),
    'max_calls_to_recovery': recovered['calls_to_recovery'].max(),
}])

,avg_calls_to_recovery,median_calls_to_recovery,min_calls_to_recovery,max_calls_to_recovery
0,3.404762,2.5,1.0,11.0


In [6]:
repair_calls = step_df[step_df['stage'] == 'repair']
plan_code_calls = step_df[step_df['stage'] == 'plan_code']
planner_calls = step_df[step_df['stage'] == 'plan']

action_summary = pd.DataFrame([
    {
        'action': 'repair',
        'calls': len(repair_calls),
        'successes': (repair_calls['status'] == 'PASS').sum(),
        'success_rate': (repair_calls['status'] == 'PASS').mean() if len(repair_calls) else 0,
        'avg_tokens': repair_calls['total_tokens'].mean() if len(repair_calls) else 0,
        'avg_latency': repair_calls['latency_sec'].mean() if len(repair_calls) else 0,
    },
    {
        'action': 'plan_code',
        'calls': len(plan_code_calls),
        'successes': (plan_code_calls['status'] == 'PASS').sum(),
        'success_rate': (plan_code_calls['status'] == 'PASS').mean() if len(plan_code_calls) else 0,
        'avg_tokens': plan_code_calls['total_tokens'].mean() if len(plan_code_calls) else 0,
        'avg_latency': plan_code_calls['latency_sec'].mean() if len(plan_code_calls) else 0,
    },
    {
        'action': 'planner_only',
        'calls': len(planner_calls),
        'successes': None,
        'success_rate': None,
        'avg_tokens': planner_calls['total_tokens'].mean() if len(planner_calls) else 0,
        'avg_latency': planner_calls['latency_sec'].mean() if len(planner_calls) else 0,
    }
])

action_summary

,action,calls,successes,success_rate,avg_tokens,avg_latency
0,repair,156,12.0,0.076923,723.538462,2.606975
1,plan_code,121,30.0,0.247934,424.958678,2.111416
2,planner_only,121,NaN,NaN,490.289256,1.557162


In [7]:
def extract_state(row):
    ps = row.get('policy_state')
    if not isinstance(ps, dict):
        return None
    return (
        ps.get('last_fail'),
        ps.get('last_error'),
        ps.get('same_signature_run_length')
    )

sa_df = step_df[step_df['stage'].isin(['repair', 'plan_code'])].copy()
sa_df['state'] = sa_df.apply(extract_state, axis=1)
sa_df = sa_df[sa_df['state'].notnull()].copy()

sa_df['success'] = (sa_df['status'] == 'PASS').astype(int)

state_action_summary = (
    sa_df
    .groupby(['state', 'stage'])
    .agg(
        count=('success', 'count'),
        success_rate=('success', 'mean'),
        avg_tokens=('total_tokens', 'mean'),
        avg_latency=('latency_sec', 'mean'),
    )
    .reset_index()
)

state_action_summary['success_per_1k_tokens'] = (
    state_action_summary['success_rate'] / 
    (state_action_summary['avg_tokens'] / 1000)
)

state_action_summary.sort_values(['state', 'stage'])

,state,stage,count,success_rate,avg_tokens,avg_latency,success_per_1k_tokens
0,"(CODE_FAIL, empty_output, 1)",repair,1,0.000000,342.000000,0.901474,0.000000
1,"(EXEC_FAIL, AssertionError, 1)",repair,1,0.000000,675.000000,3.317653,0.000000
2,"(EXEC_FAIL, AttributeError, 1)",repair,1,0.000000,1064.000000,4.957435,0.000000
3,"(EXEC_FAIL, IndentationError, 1)",repair,2,0.000000,836.000000,0.865564,0.000000
4,"(EXEC_FAIL, NameError, 1)",repair,20,0.000000,1121.900000,5.043260,0.000000
5,"(EXEC_FAIL, NameError, 2)",plan_code,13,0.692308,382.230769,1.154374,1.811230
6,"(EXEC_FAIL, SyntaxError, 1)",repair,59,0.000000,785.610169,2.817944,0.000000
7,"(EXEC_FAIL, SyntaxError, 2)",plan_code,3,0.000000,525.000000,2.229213,0.000000
8,"(EXEC_FAIL, SyntaxError, 3)",repair,1,0.000000,528.000000,1.329230,0.000000
9,"(EXEC_FAIL, TypeError, 1)",repair,1,1.000000,676.000000,2.888103,1.479290


In [8]:
core_summary = pd.DataFrame([{
    'method': METHOD_LABELS.get('rpe_policy', 'rpe_policy'),
    'total_problems': len(traj_df),
    'initial_failed': n_initial_failed,
    'recovery_rate': recovery_rate,
    'unresolved_rate': unresolved_rate,
    'avg_calls_to_recovery': recovered['calls_to_recovery'].mean(),
    'median_calls_to_recovery': recovered['calls_to_recovery'].median(),
    'repair_calls': len(repair_calls),
    'repair_success_rate': (repair_calls['status'] == 'PASS').mean() if len(repair_calls) else 0,
    'plan_code_calls': len(plan_code_calls),
    'plan_code_success_rate': (plan_code_calls['status'] == 'PASS').mean() if len(plan_code_calls) else 0,
    'avg_total_calls': traj_df['call_count'].mean(),
    'final_pass_rate': (traj_df['final_status'] == 'PASS').mean(),
}])

core_summary

,method,total_problems,initial_failed,recovery_rate,unresolved_rate,avg_calls_to_recovery,median_calls_to_recovery,repair_calls,repair_success_rate,plan_code_calls,plan_code_success_rate,avg_total_calls,final_pass_rate
0,Policy Loop,164,55,0.763636,0.236364,3.404762,2.5,156,0.076923,121,0.247934,3.426829,0.920732
